# Other MVP choices: data, inference, telemetry, evaluation and interface

This notebook compares the remaining choices from the project brief. It reads existing result files and makes no new model calls. Alternatives shown as examples below are clearly marked; the brief only requires “or alternatives” and “or something similar,” without naming a specific alternative framework or UI.

## Corpus creation: curate versus scrape

| | Manually curate | Scrape |
|---|---|---|
| Work required | Select and normalize each recipe; verify fields and labels | Obtain sources, check permission/terms, extract, clean and deduplicate |
| Strength | Easier to inspect provenance and quality | Can gather a larger or more varied candidate set |
| Risk | Small, biased or synthetic collection | Extraction errors, source restrictions, inconsistent or missing labels |
| Current project | 30 AI-assisted synthetic recipes, not independently checked | No scraper implemented |

Both approaches need ingredient, instruction, time and dietary fields, with human review. Synthetic estimated times are not verified cooking times. For either approach, retain source provenance and audit allergen claims.

## Inference and telemetry

The brief specifies Groq’s free tier with an open-weights model and gives Llama 3.1 8B as an example. Groq retired that example for free/developer tiers, so V2 uses `openai/gpt-oss-20b`; availability and quotas depend on the account. The brief does not name a second MVP model to benchmark. Do not claim a speed comparison between models until both are actually tested with the same prompts and settings.

Input and output tokens measure provider-reported usage across the model calls; total request latency includes routing, retrieval, tool calls and generation. TTFT measures the wait to the first nonempty final-answer token and is not the same as total latency. The app reports unknown token counts as missing rather than zero. Model-generated reasoning tokens and provider billing can differ from simple input/output displays.

## Orchestration, MCP and UI

LlamaIndex handles document indexing/retrieval; LangGraph controls state and routes. Their responsibilities are different and they can be combined, as in V2. A different framework is permitted by the brief but none is named as a required comparison. MCP tools perform operations and resources expose context; see notebook 11 for a runnable comparison of the actual patterns and client approaches. Streamlit is the named UI example; an equivalent local interface is allowed, but the brief does not name a specific second UI. Compare candidates using the same required features: conversation, source chunks, MCP calls, token usage and latency. No alternative UI has been benchmarked here.

## Ragas: three distinct questions

| Metric | Question answered | What a low score suggests | Limitation |
|---|---|---|---|
| Context precision | Are relevant chunks ranked ahead of irrelevant ones? | Inspect chunk boundaries, retrieval and ranking | Depends on the evaluation reference and judge |
| Context recall | Was the evidence needed for the reference retrieved? | Inspect corpus coverage, chunking, embedding and top-k | Does not prove answer generation is correct |
| Faithfulness | Are answer claims supported by the retrieved context? | Inspect the answer and grounding instruction | A faithful answer can still reflect an incorrect source |

All three are judge-based signals, not ground truth. The test data and reference answers in this project need independent review. Questions expecting a lack of a substitution cannot establish absence across the whole corpus from top-k retrieval alone.


In [ ]:
from pathlib import Path
import json
import pandas as pd
HERE=Path.cwd().resolve()
if HERE.name != 'comparisons_v1': raise RuntimeError('Start this notebook in v2/comparisons_v1')
V2=HERE.parent
run_path=V2/'results'/'evaluation_sections_k5_20261125_130014.json'
if not run_path.exists(): raise FileNotFoundError('Expected saved evaluation run is not present; run v2/evaluate.py first and update run_path.')
run=json.loads(run_path.read_text(encoding='utf-8'))
references={x['id']:x for x in json.loads((V2/'data'/'evaluation_v2.json').read_text(encoding='utf-8'))}
records=[]
for case in run['cases']:
    row={'id':case['id'], 'expected_route':case['expected_route'], 'actual_route':case.get('actual_route'), 'route_correct':case.get('route_correct',False), 'error':case.get('error')}
    expected=references[case['id']].get('expected_tool_result')
    if expected is not None:
        try: row['tool_correct']=json.loads(case.get('tool_result') or 'null')==expected
        except (TypeError, ValueError): row['tool_correct']=False
    records.append(row)
routes=pd.DataFrame(records)
display(routes)
print('Expected routes:', int(routes.route_correct.sum()), '/', len(routes))
if 'tool_correct' in routes:
    scored=routes.tool_correct.dropna()
    print('Expected tool results:', int(scored.sum()), '/', len(scored))


## Inspect the actual saved Ragas batches

Only batches whose `source_run` matches the selected run are combined. If a case was scored more than once, the most recently modified batch wins; a more recent failed result can replace an earlier score, so inspect the table and the source files. Missing metrics remain missing, not zero. The saved 20-question run predates the latest answer-prompt improvement, so its scores do not measure that newer prompt. Results may be incomplete because Groq returned rate-limit errors.


In [ ]:
metric_names=['llm_context_precision_with_reference','context_recall','faithfulness']
scores={}
for p in sorted((V2/'results').glob('ragas_batch_*.json'),key=lambda x:x.stat().st_mtime):
    batch=json.loads(p.read_text(encoding='utf-8'))
    if Path(batch.get('source_run','')).name != run_path.name: continue
    for case in batch.get('cases',[]):
        raw=case.get('scores',{})
        scores[case['id']]={'id':case['id'], **{name:raw.get(name) for name in metric_names}, 'error':case.get('error'), 'batch_file':p.name}
table=pd.DataFrame([scores.get(case['id'], {'id':case['id']}) for case in run['cases'] if case['expected_route']=='retrieve'])
display(table)
for name in metric_names:
    if name in table:
        vals=pd.to_numeric(table[name],errors='coerce')
        print(f'{name}: {vals.notna().sum()}/{len(table)} scored; mean of available scores = {vals.mean():.3f}' if vals.notna().any() else f'{name}: no scores')
print('Do not compare these partial means with a full benchmark or treat missing scores as zero.')
